In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/balraj98/massachusetts-roads-dataset"):
    print(root, "->", files[:5])

/kaggle/input/datasets/balraj98/massachusetts-roads-dataset -> ['label_class_dict.csv', 'metadata.csv']
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff -> []
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff/val_labels -> ['10978735_15.tif', '10228690_15.tif', '24328810_15.tif', '22528900_15.tif', '25229230_15.tif']
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff/test_labels -> ['26578720_15.tif', '23278930_15.tif', '10378780_15.tif', '22078975_15.tif', '11278840_15.tif']
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff/val -> ['21929020_15.tiff', '10978735_15.tiff', '24328810_15.tiff', '23128930_15.tiff', '22829035_15.tiff']
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff/train_labels -> ['24778855_15.tif', '24928900_15.tif', '10828825_15.tif', '21778960_15.tif', '25979290_15.tif']
/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff/test -> ['11278840_15.tiff', '18328735_15.tiff', '24628885_

In [ ]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
import glob
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
 
DATA_DIR = "/kaggle/input/datasets/balraj98/massachusetts-roads-dataset/tiff"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TRAIN_MASK_DIR = os.path.join(DATA_DIR, "train_labels")
VAL_IMG_DIR = os.path.join(DATA_DIR, "val")
VAL_MASK_DIR = os.path.join(DATA_DIR, "val_labels")
 
IMG_SIZE = 384
IN_CHANNELS = 3
N_CLASSES = 1           
NFILTERS_INIT = 32
DEPTH = 5
BATCH_SIZE = 4           
NUM_EPOCHS = 60
LR = 3e-4                 
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
SEED = 42
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RMSNorm1D(nn.Module):
    """RMSNorm over the channel dim of a pooled (B, C) vector."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
 
    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).sqrt()
        return x / (rms + self.eps) * self.weight
 
 
class AttnResBlockA(nn.Module):
    def __init__(self, in_ch, out_ch, dilation_rates=(1, 3, 15, 31)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.BatchNorm2d(in_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=d, dilation=d, bias=False),
            )
            for d in dilation_rates
        ])
        self.proj = None
        if in_ch != out_ch:
            self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
 
        
        self.query = nn.Parameter(torch.zeros(out_ch))
        self.key_norm = RMSNorm1D(out_ch)
 
    def forward(self, x):
        identity = x if self.proj is None else self.proj(x)
        branch_outs = [branch(x) for branch in self.branches]
        sources = [identity] + branch_outs          
 
        
        keys = [self.key_norm(s.mean(dim=(2, 3))) for s in sources]
 
        
        logits = torch.stack(
            [torch.einsum('bc,c->b', k, self.query) for k in keys], dim=1
        ) 
        alpha = torch.softmax(logits, dim=1) 
 
        out = 0
        for i, s in enumerate(sources):
            out = out + alpha[:, i].view(-1, 1, 1, 1) * s
        return out
 
class PSPPooling(nn.Module):
    def __init__(self, channels, pool_sizes=(1, 2, 4, 8)):
        super().__init__()
        self.pool_sizes = pool_sizes
        branch_ch = channels // len(pool_sizes)
        self.in_proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, branch_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            )
            for _ in pool_sizes
        ])
        self.out_conv = nn.Sequential(
            nn.Conv2d(branch_ch * len(pool_sizes) + channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        h, w = x.shape[-2:]
        base = self.in_proj(x)
        feats = [base]
        for pool_size, branch in zip(self.pool_sizes, self.branches):
            pooled = F.adaptive_avg_pool2d(x, output_size=pool_size)
            proj = branch(pooled)
            up = F.interpolate(proj, size=(h, w), mode="bilinear", align_corners=False)
            feats.append(up)
        return self.out_conv(torch.cat(feats, dim=1))

 
class DownStage(nn.Module):
    def __init__(self, in_ch, out_ch, dilation_rates):
        super().__init__()
        self.resblock = AttnResBlockA(in_ch, out_ch, dilation_rates)
        self.downsample = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=2, padding=1)
 
    def forward(self, x):
        skip = self.resblock(x)
        down = self.downsample(skip)
        return skip, down
 
 
class UpStage(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dilation_rates):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.resblock = AttnResBlockA(out_ch + skip_ch, out_ch, dilation_rates)
 
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.resblock(x)
 
 
class ResUNetA(nn.Module):
    def __init__(self, in_channels=3, n_classes=1, nfilters_init=32, depth=6):
        super().__init__()
        self.depth = depth
 
        def rates_for_stage(stage_idx):
            schedule = [(1, 3, 15, 31), (1, 3, 15, 31), (1, 3, 15),
                        (1, 3, 15), (1, 3), (1,), (1,)]
            return schedule[min(stage_idx, len(schedule) - 1)]
 
        self.stem = nn.Conv2d(in_channels, nfilters_init, kernel_size=3, padding=1)
 
        self.down_stages = nn.ModuleList()
        ch = nfilters_init
        enc_channels = [ch]
        for i in range(depth):
            out_ch = ch * 2
            self.down_stages.append(DownStage(ch, out_ch, rates_for_stage(i)))
            ch = out_ch
            enc_channels.append(ch)
 
        self.bridge_resblock = AttnResBlockA(ch, ch, rates_for_stage(depth))
        self.bridge_psp = PSPPooling(ch)
 
        self.up_stages = nn.ModuleList()
        for i in range(depth):
            skip_ch = enc_channels[depth - i]
            out_ch = enc_channels[depth - i - 1]
            rates = rates_for_stage(depth - i - 1)
            self.up_stages.append(UpStage(ch, skip_ch, out_ch, rates))
            ch = out_ch
 
        self.combine_psp = PSPPooling(ch)
        self.head = nn.Conv2d(ch, n_classes, kernel_size=1)
 
    def forward(self, x):
        x = self.stem(x)
 
        skips = []
        for stage in self.down_stages:
            skip, x = stage(x)
            skips.append(skip)
 
        x = self.bridge_resblock(x)
        x = self.bridge_psp(x)
 
        for stage, skip in zip(self.up_stages, reversed(skips)):
            x = stage(x, skip)
 
        x = self.combine_psp(x)
        return self.head(x)
 
 
class TanimotoLoss(nn.Module):
    def __init__(self, smooth=1e-3):
        super().__init__()
        self.smooth = smooth
 
    def _tanimoto(self, p, y):
        dims = tuple(range(1, p.dim()))
        inter = (p * y).sum(dim=dims)
        p_sq = (p * p).sum(dim=dims)
        y_sq = (y * y).sum(dim=dims)
        return (inter + self.smooth) / (p_sq + y_sq - inter + self.smooth)
 
    def forward(self, logits, target):
        if logits.shape[1] == 1:
            p = torch.sigmoid(logits)
        else:
            p = torch.softmax(logits, dim=1)
 
        t1 = self._tanimoto(p, target)
        t2 = self._tanimoto(1 - p, 1 - target)
        loss = 1 - 0.5 * (t1 + t2)
        return loss.mean()
 
 
class RoadsDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
 
    def __len__(self):
        return len(self.image_paths)
 
    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
 
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  
 
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]
 
        mask = mask.unsqueeze(0) if mask.dim() == 2 else mask  
        return image, mask
 
 
def build_transforms():
    train_tf = A.Compose([
        A.RandomCrop(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    val_tf = A.Compose([
        A.CenterCrop(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train_tf, val_tf

 
@torch.no_grad()
def iou_score(logits, target, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    union = (preds + target).clamp(0, 1).sum(dim=(1, 2, 3))
    return ((inter + eps) / (union + eps)).mean().item()
 
 
@torch.no_grad()
def dice_score(logits, target, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    denom = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2 * inter + eps) / (denom + eps)).mean().item()
 
 
def train_one_epoch(model, loader, loss_fn, optimizer, scaler, device):
    model.train()
    running_loss, running_iou, running_dice = 0.0, 0.0, 0.0
 
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
 
        optimizer.zero_grad(set_to_none=True)
 
        with autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)
 
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  [warning] non-finite loss encountered ({loss.item()}), skipping this batch")
            optimizer.zero_grad(set_to_none=True)
            del logits, loss
            torch.cuda.empty_cache()
            continue
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
 
        running_loss += loss.item() * images.size(0)
        running_iou += iou_score(logits, masks) * images.size(0)
        running_dice += dice_score(logits, masks) * images.size(0)
 
    n = len(loader.dataset)
    return running_loss / n, running_iou / n, running_dice / n
 
 
@torch.no_grad()
def validate(model, loader, loss_fn, device):
    model.eval()
    running_loss, running_iou, running_dice = 0.0, 0.0, 0.0
 
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
 
        with autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)
 
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  [warning] non-finite val loss encountered ({loss.item()}), skipping this batch")
            continue
 
        running_loss += loss.item() * images.size(0)
        running_iou += iou_score(logits, masks) * images.size(0)
        running_dice += dice_score(logits, masks) * images.size(0)
 
    n = len(loader.dataset)
    return running_loss / n, running_iou / n, running_dice / n

 
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
 
    train_images = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "*")))
    train_masks = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, "*")))
    val_images = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*")))
    val_masks = sorted(glob.glob(os.path.join(VAL_MASK_DIR, "*")))
 
    assert len(train_images) == len(train_masks) and len(train_images) > 0, \
        "Mismatched or empty train image/mask lists — check TRAIN_IMG_DIR / TRAIN_MASK_DIR paths."
    assert len(val_images) == len(val_masks) and len(val_images) > 0, \
        "Mismatched or empty val image/mask lists — check VAL_IMG_DIR / VAL_MASK_DIR paths."
 
    for ip, mp in zip(train_images[:5], train_masks[:5]):
        ip_id = os.path.splitext(os.path.basename(ip))[0]
        mp_id = os.path.splitext(os.path.basename(mp))[0]
        assert ip_id == mp_id, f"Train image/mask id mismatch: {ip_id} vs {mp_id}"
 
    train_tf, val_tf = build_transforms()
    train_ds = RoadsDataset(train_images, train_masks, transform=train_tf)
    val_ds = RoadsDataset(val_images, val_masks, transform=val_tf)
 
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
 
    print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
 
    model = ResUNetA(in_channels=IN_CHANNELS, n_classes=N_CLASSES,
                      nfilters_init=NFILTERS_INIT, depth=DEPTH).to(DEVICE)
    loss_fn = TanimotoLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = GradScaler()
 
    best_val_iou = 0.0
    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_iou, train_dice = train_one_epoch(
            model, train_loader, loss_fn, optimizer, scaler, DEVICE)
        val_loss, val_iou, val_dice = validate(model, val_loader, loss_fn, DEVICE)
        scheduler.step()
        dt = time.time() - t0
 
        print(f"Epoch {epoch:03d}/{NUM_EPOCHS} | {dt:.1f}s | "
              f"train_loss={train_loss:.4f} train_iou={train_iou:.4f} train_dice={train_dice:.4f} | "
              f"val_loss={val_loss:.4f} val_iou={val_iou:.4f} val_dice={val_dice:.4f} | "
              f"lr={scheduler.get_last_lr()[0]:.2e}")
 
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            ckpt_path = os.path.join(CHECKPOINT_DIR, "resuneta_attnres_massroads_best.pt")
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_iou": val_iou,
            }, ckpt_path)
            print(f"  -> saved new best checkpoint (val_iou={val_iou:.4f})")
 
    print(f"Training complete. Best val IoU: {best_val_iou:.4f}")
 
 
if __name__ == "__main__":
    main()

Train samples: 1108 | Val samples: 14


/tmp/ipykernel_58/3768657638.py:379: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/3768657638.py:293: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this batch
  [warning] non-finite loss encountered (nan), skipping this b